## How to Run Borealis 2 Preview Version (Quantized Q4_K_M)

Nasjonalbiblioteket has released a preview version of Borealis 2, pre-trained on a large mix of Nordic-language sources, including licensed material such as newspaper text. According to the model card, the model is based on google/gemma-4-26B-A4B-it, which then underwent continued pre-training on Aurora 2604 data and supervised fine-tuning on textual instructions from NbAiLab/aurora-sft-2606. The model card can be found [here](https://huggingface.co/NbAiLab/borealis2-26b-a4b-preview-gguf).

> **Note:** This is a preview experiment. The model card states that it has not been fully safety-aligned and should not be used for safety-critical applications. Treat its output as experimental and verify anything important.

### Setup

To run a quantized GGUF version of Borealis 2, you can use either a good local machine or a Colab GPU. This notebook uses Colab, with the purpose of showing users how to access the model regardless of their own hardware. To get access to a GPU in Colab you need either a Pro subscription or Pay-as-you-go for the compute units (more [here](https://colab.research.google.com/signup)).

For this code to work you need to be logged in to your Google account, and connected to at least an L4 GPU. (Even the quantized preview model is too large to run on the free CPU in Colab.)

### Step-by-Step

The purpose of this notebook is to set up a connection with Borealis 2 Q4_K_M, ask a test prompt to check that inference works, then run a test set of 30 prompts. The test set is designed to get a quick sense of how Borealis 2 performs on prompts specifically related to Norwegian and Norway.

Step-by-step plan:

1. Set up access to Borealis 2 Q4_K_M from HuggingFace.
2. Test that inference works with a test prompt.
3. Load a test set from JSON and run a 30-prompt test set.

Let's get started!

### 1. Set up access to Borealis 2 Q4_K_M from HuggingFace

In [1]:
# Environment setup: Google Colab (GPU runtime)
import os, sys, subprocess

def _pip(*args: str) -> None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

# ── Runtime info ──
print("=" * 48)
print(f"  KERNEL: COLAB (cloud runtime)")
print(f"  Python: {sys.version.split()[0]}")
print(f"  Path:   {sys.executable}")
print("=" * 48)

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()
print(f"GPU: {gpu or 'none detected — check Runtime > Change runtime type'}")

from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/borealis2"   # data + outputs → persists on Drive
MODEL_DIR   = "/content/models"                    # weights → local VM disk

# Make sure both exist
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"PROJECT_DIR = {PROJECT_DIR}")
print(f"MODEL_DIR   = {MODEL_DIR}")

In [3]:
# Install Llama CPP on the virtual machine
_pip("llama-cpp-python",
     "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu124")

In [4]:
# Create a connection with the model using Llama CPP
from llama_cpp import Llama

llm = Llama.from_pretrained(
    repo_id="NbAiLab/borealis2-26b-a4b-preview-gguf",
    filename="*Q4_K_M.gguf",
    cache_dir=MODEL_DIR,
    n_gpu_layers=-1,
    n_ctx=4096,
    verbose=True,
)

### 2. Test that inference works with a test prompt

In [5]:
# Test prompt checking connection to the model
out = llm.create_chat_completion(
    messages=[
        {"role": "user", "content": "Hva jobber Nasjonalbiblioteket med?"}
    ],
    max_tokens=2048,   # matches the batch run; n_ctx=4096 leaves plenty of headroom
    temperature=0.7,   # same as the batch run, so the smoke test is representative
)

print(out["choices"][0]["message"]["content"])

### 3. Load a test set from JSON and run 30 prompt test set

Now it's time to load the test set from a JSON file we have stored in the project directory. Model responses get saved to a new JSON file in the same folder.

In [6]:
# Running 30 prompt test set
import json, os, time, datetime

TEST_FILE = os.path.join(PROJECT_DIR, "borealis_test_prompts_v1.json")

assert os.path.exists(TEST_FILE), (
    f"Test set not found at {TEST_FILE}. "
    f"Upload borealis_test_prompts_v1.json to {PROJECT_DIR} on your Drive first."
)

with open(TEST_FILE, encoding="utf-8") as f:
    test_prompts = json.load(f)

# Generation settings, tweak as you like
GEN_KWARGS = dict(max_tokens=2048, temperature=0.7, top_p=0.95)

results = []
run_started = datetime.datetime.now().isoformat(timespec="seconds")

for i, item in enumerate(test_prompts, 1):
    print(f"[{i}/{len(test_prompts)}] id={item['id']} ({item['category']})")
    t0 = time.time()
    try:
        out = llm.create_chat_completion(
            messages=[{"role": "user", "content": item["prompt"]}],
            **GEN_KWARGS,
        )
        response = out["choices"][0]["message"]["content"]
        usage = out.get("usage")
        error = None
    except Exception as e:
        response, usage, error = None, None, repr(e)
        print(f"    ERROR: {error}")

    results.append({
        **item,                                   # all original fields (id, category, prompt)
        "response": response,
        "error": error,
        "usage": usage,
        "elapsed_sec": round(time.time() - t0, 2),
    })

# Save
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = os.path.join(PROJECT_DIR, f"borealis2_results_{stamp}.json")

payload = {
    "model": "NbAiLab/borealis2-26b-a4b-preview-gguf",
    "source_file": os.path.basename(TEST_FILE),
    "run_started": run_started,
    "run_finished": datetime.datetime.now().isoformat(timespec="seconds"),
    "gen_kwargs": GEN_KWARGS,
    "results": results,
}

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print(f"\nSaved {len(results)} results -> {out_path}")

In [7]:
# Close Colab session
from google.colab import runtime
runtime.unassign()

Now you should have your model responses to the test set in your project folder, ready for inspection. Remember to also do a quick check in Colab that you have disconnected and have no active sessions for this notebook (Runtime -> Manage sessions).

To run your own prompts you can create your own test set using the same JSON format. Feel free to contact me if you need any help!